In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
words = ["paper", "Papier", "papel", "palpiri", "carta", "karatasi"]

In [ ]:
def edit_distance(a, b):
    """Levenshtein distance via DP."""
    a, b = a.lower(), b.lower()
    m, n = len(a), len(b)
    dp = list(range(n + 1))
    for i, ca in enumerate(a, 1):
        prev = dp[:]
        dp[0] = i
        for j, cb in enumerate(b, 1):
            dp[j] = prev[j-1] if ca == cb else 1 + min(prev[j], dp[j-1], prev[j-1])
    return dp[n]



In [ ]:
distances = pd.DataFrame(
    [[edit_distance(w1, w2) for w2 in words] for w1 in words],
    index=words,
    columns=words,
)
distances

In [ ]:
def classical_mds(D, k=3):
    n = len(D)
    D2 = np.array(D) ** 2
    H = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * H @ D2 @ H

    eigenvalues, eigenvectors = np.linalg.eigh(B)
    eigenvalues, eigenvectors = eigenvalues[::-1], eigenvectors[:, ::-1]

    # Drop negative eigenvalues (edit distance is not Euclidean)
    k_pos = (eigenvalues > 1e-10).sum()
    k_use = min(k, k_pos)
    coords = eigenvectors[:, :k_use] * np.sqrt(eigenvalues[:k_use])
    if k_use < k:
        coords = np.hstack([coords, np.zeros((n, k - k_use))])
    return coords


def lab_to_rgb(Lab):
    """Convert an (n, 3) array of CIE L*a*b* values to sRGB in [0, 1]."""
    L, a, b = Lab[:, 0], Lab[:, 1], Lab[:, 2]
    fy = (L + 16) / 116
    fx = a / 500 + fy
    fz = fy - b / 200

    def f_inv(t):
        return np.where(t > 6 / 29, t ** 3, 3 * (6 / 29) ** 2 * (t - 4 / 29))

    XYZ = np.stack([0.95047 * f_inv(fx), f_inv(fy), 1.08883 * f_inv(fz)], axis=1)

    # XYZ → linear sRGB
    M = np.array([[ 3.2406, -1.5372, -0.4986],
                  [-0.9689,  1.8758,  0.0415],
                  [ 0.0557, -0.2040,  1.0570]])
    rgb_lin = XYZ @ M.T

    # Gamma correction (sRGB)
    rgb = np.where(rgb_lin <= 0.0031308,
                   12.92 * rgb_lin,
                   1.055 * rgb_lin ** (1 / 2.4) - 0.055)
    return np.clip(rgb, 0, 1)

In [ ]:
coords = classical_mds(distances)

# Uniform scaling: one scale factor for all dims so MDS distances are preserved.
# Per-dimension normalization would amplify near-zero eigenvalue dimensions,
# blowing up tiny coordinate differences into large color differences.
target_half_ranges = np.array([25, 50, 50])  # max half-extents for L, a, b
uniform_scale = (target_half_ranges / np.abs(coords).max(axis=0)).min()

Lab = coords * uniform_scale + np.array([55, 0, 0])  # center L at 55

colors = lab_to_rgb(Lab)

pd.DataFrame(Lab, index=words, columns=["L", "a", "b"]).round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2))

for i, (word, color) in enumerate(zip(words, colors)):
    ax.scatter(i, 0.15, color=color, s=500, zorder=3, edgecolors="0.3", linewidths=0.5)
    ax.text(i, -0.05, word, ha="center", va="top", fontsize=12)

ax.set_xlim(-0.7, len(words) - 0.3)
ax.set_ylim(-0.3, 0.45)
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import jax
import jax.numpy as jnp
from vizopt.jaxopt import optimize_gradient_descent


def rgb_to_lab(rgb):
    """Differentiable sRGB → CIE L*a*b* (D65). rgb: (n, 3) in [0, 1]."""
    # sRGB → linear RGB
    rgb_lin = jnp.where(rgb <= 0.04045, rgb / 12.92, ((rgb + 0.055) / 1.055) ** 2.4)
    # linear RGB → XYZ (D65)
    M = jnp.array([[0.4124564, 0.3575761, 0.1804375],
                   [0.2126729, 0.7151522, 0.0721750],
                   [0.0193339, 0.1191920, 0.9503041]])
    xyz = (rgb_lin @ M.T) / jnp.array([0.95047, 1.00000, 1.08883])
    # XYZ → LAB
    delta = 6 / 29
    f = jnp.where(xyz > delta ** 3, xyz ** (1 / 3), xyz / (3 * delta ** 2) + 4 / 29)
    L = 116 * f[:, 1] - 16
    a = 500 * (f[:, 0] - f[:, 1])
    b = 200 * (f[:, 1] - f[:, 2])
    return jnp.stack([L, a, b], axis=1)

In [ ]:
# Pre-compute pair indices and target distances (fixed constants, not optimized)
n = len(words)
_pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
idx_i = jnp.array([i for i, j in _pairs])
idx_j = jnp.array([j for i, j in _pairs])

D = np.array(distances)
edit_pairs = jnp.array([D[i, j] for i, j in _pairs], dtype=float)

# Fix the target scale: most-distant word pair gets ΔE = target_max_delta_e.
# A free scale parameter would let the optimizer collapse all colors to gray.
target_max_delta_e = 50.0
edit_targets = edit_pairs / edit_pairs.max() * target_max_delta_e

def loss_fn(params):
    rgb = jax.nn.sigmoid(params["logit_rgb"])
    lab = rgb_to_lab(rgb)
    color_dists = jnp.sqrt(((lab[idx_i] - lab[idx_j]) ** 2).sum(axis=-1) + 1e-8)

    # Stress: match pairwise ΔE to scaled edit distances
    stress = jnp.mean((color_dists - edit_targets) ** 2)

    # Coverage: maximize variance of a and b (chroma channels).
    # Without this, any gray solution that satisfies the stress term is equally valid.
    coverage = -(lab[:, 1].var() + lab[:, 2].var())

    return stress + 0.5 * coverage

# Warm-start from MDS result
rgb_init = np.clip(lab_to_rgb(Lab), 1e-4, 1 - 1e-4)
logit_init = np.log(rgb_init / (1 - rgb_init))
params0 = {"logit_rgb": jnp.array(logit_init)}

sparse_cb = lambda i, loss, *_: print(f"iter {i:4d}  loss={loss:.6f}") if i % 200 == 0 else None

params_opt = optimize_gradient_descent(
    params0, loss_fn, learning_rate=0.05, n_iters=1000, callback=sparse_cb
)

colors = np.array(jax.nn.sigmoid(params_opt["logit_rgb"]))

In [ ]:
from scipy.stats import spearmanr

lab_opt = np.array(rgb_to_lab(jnp.array(colors)))
color_dists_opt = [np.linalg.norm(lab_opt[i] - lab_opt[j]) for i, j in _pairs]
rho, _ = spearmanr(color_dists_opt, [D[i, j] for i, j in _pairs])
print(f"Spearman ρ = {rho:.3f}")

fig, ax = plt.subplots(figsize=(8, 2))
for i, (word, color) in enumerate(zip(words, colors)):
    ax.scatter(i, 0.15, color=color, s=500, zorder=3, edgecolors="0.3", linewidths=0.5)
    ax.text(i, -0.05, word, ha="center", va="top", fontsize=12)
ax.set_xlim(-0.7, len(words) - 0.3)
ax.set_ylim(-0.3, 0.45)
ax.axis("off")
plt.tight_layout()
plt.show()